In [8]:
import sys
import subprocess
import os

# Force bitsandbytes to use CUDA 12.1 backend to avoid missing libnvJitLink.so.13 errors on Kaggle/Colab
os.environ["BNB_CUDA_VERSION"] = "121"

print("Installing pipeline dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)

print("Uninstalling potentially conflicting pre-installed packages to avoid binary mismatch...")
for pkg in ["numpy", "pandas", "pyarrow", "scipy", "datasets"]:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", pkg], check=False)
    # Run a second time to ensure shadowed user-site packages are also removed
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", pkg], check=False)

libs = [
    "numpy==1.26.4",
    "scipy<1.13.0",
    "pandas<2.2.0",
    "pyarrow<15.0.0",
    "torch",
    "torchvision",
    "torchaudio",
    "torchao",
    "bitsandbytes>=0.45.3",
    "diffusers[torch]>=0.33.0",
    "transformers>=4.48.0",
    "accelerate>=0.30.0",
    "peft>=0.14.0",
    "sentencepiece",
    "imageio",
    "imageio-ffmpeg",
    "safetensors",
    "Pillow==11.3.0",
    "tqdm",
    "datasets",
    "deepspeed",
    "decord",
    "av",
    "einops",
    "toml",
    "loguru",
    "ftfy"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall"] + libs + ["--extra-index-url", "https://download.pytorch.org/whl/cu121"], check=True)

# Reload numpy, pandas, and PIL in the current session to pick up the freshly installed C binaries.
# Without this, Python keeps old (stale) modules loaded in memory, causing binary or typing incompatibility
import importlib
for mod_name in list(sys.modules.keys()):
    if mod_name == "numpy" or mod_name.startswith("numpy."):
        del sys.modules[mod_name]
for mod_name in list(sys.modules.keys()):
    if mod_name == "pandas" or mod_name.startswith("pandas."):
        del sys.modules[mod_name]
for mod_name in list(sys.modules.keys()):
    if mod_name == "PIL" or mod_name.startswith("PIL."):
        del sys.modules[mod_name]
import numpy as np
print(f"numpy {np.__version__} loaded successfully (dtype size = {np.dtype('float64').itemsize * 12} bits).")

# Inject _Ink compatibility guard if needed for PIL._typing
try:
    import PIL
    import PIL._typing
    if not hasattr(PIL._typing, "_Ink"):
        from typing import Union, Tuple, Any
        PIL._typing._Ink = Union[Tuple[int, ...], str, int, float, Any]
    from PIL import Image
    print(f"Pillow {PIL.__version__} verified and working properly!")
except Exception as e:
    print(f"PIL verification notice: {e}")

# Clone diffusion-pipe with submodules (for comfy library imports)
if not os.path.exists("diffusion-pipe"):
    print("Cloning diffusion-pipe recursively...")
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/tdrussell/diffusion-pipe.git"], check=True)
else:
    print("diffusion-pipe directory already exists. Ensuring submodules are up to date...")
    subprocess.run(["git", "-C", "diffusion-pipe", "submodule", "update", "--init", "--recursive"], check=True)

# Programmatically patch attention.py to support SDPA fallback on Kaggle T4 GPUs
attention_file = "diffusion-pipe/models/wan/attention.py"
if os.path.exists(attention_file):
    print("Applying FlashAttention fallback patch to attention.py...")
    with open(attention_file, "r", encoding="utf-8") as f:
        content = f.read().replace('\r\n', '\n')
    
    if "scaled_dot_product_attention" in content and "if not (FLASH_ATTN_2_AVAILABLE or FLASH_ATTN_3_AVAILABLE):" in content:
        print("attention.py is already patched!")
    else:
        # Find flash_attention function definition and insert the SDPA check at the start
        start_idx = content.find("def flash_attention(")
        if start_idx != -1:
            end_idx = content.find("):", start_idx)
            if end_idx != -1:
                insert_pos = end_idx + 2
                patch_code = """\n    if not (FLASH_ATTN_2_AVAILABLE or FLASH_ATTN_3_AVAILABLE):
        if q_lens is not None or k_lens is not None:
            import warnings
            warnings.warn(
                'Padding mask is disabled when using scaled_dot_product_attention. It can have a significant impact on performance.'
            )
        attn_mask = None
        
        q_dtype = q.dtype
        q = q.transpose(1, 2).to(dtype)
        k = k.transpose(1, 2).to(dtype)
        v = v.transpose(1, 2).to(dtype)
 
        if q_scale is not None:
            q = q * q_scale
 
        out = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask, is_causal=causal, scale=softmax_scale, dropout_p=dropout_p)
 
        out = out.transpose(1, 2).contiguous().to(q_dtype)
        return out"""
                new_content = content[:insert_pos] + patch_code + content[insert_pos:]
                with open(attention_file, "w", encoding="utf-8") as f:
                    f.write(new_content)
                print("Patch successfully applied to attention.py!")
            else:
                print("Error: Could not find closing parentheses of flash_attention signature.")
        else:
            print("Error: Could not find def flash_attention definition in attention.py.")
else:
    print("Warning: attention.py not found in diffusion-pipe models.")

# Patch diffusion-pipe's AdamW8bitKahan to handle missing 'percentile_clipping' key in newer bitsandbytes
adamw_patch_file = "diffusion-pipe/optimizers/adamw_8bit.py"
if os.path.exists(adamw_patch_file):
    with open(adamw_patch_file, "r", encoding="utf-8") as f:
        adamw_content = f.read()
    if 'config["percentile_clipping"]' in adamw_content:
        adamw_content = adamw_content.replace(
            'config["percentile_clipping"]',
            'config.get("percentile_clipping", 100)'
        )
        with open(adamw_patch_file, "w", encoding="utf-8") as f:
            f.write(adamw_content)
        print("Patched adamw_8bit.py: percentile_clipping KeyError fix applied.")

print("Environment Setup Complete!") 

Installing pipeline dependencies...
Uninstalling potentially conflicting pre-installed packages to avoid binary mismatch...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
nilearn 0.13.1 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
tpot 1.1.0 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
mne 1.12.1 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-col

numpy 1.26.4 loaded successfully (dtype size = 96 bits).
Pillow 11.3.0 verified and working properly!
diffusion-pipe directory already exists. Ensuring submodules are up to date...
Applying FlashAttention fallback patch to attention.py...
attention.py is already patched!
Environment Setup Complete!


In [9]:
pip install comfy-aimdo comfy-kitchen

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install comfy-aimdo

Note: you may need to restart the kernel to use updated packages.


In [10]:
from huggingface_hub import snapshot_download
import os
import torch

model_dir = "/tmp/models/wan2.1"
os.makedirs(model_dir, exist_ok=True)

print(f"Downloading Wan2.1-T2V-1.3B Native weights to {model_dir}...")
snapshot_download(
    repo_id="Wan-AI/Wan2.1-T2V-1.3B",
    local_dir=model_dir,
    ignore_patterns=["*.md", "*.git*", "*.txt"],
    max_workers=4
)
print("Base model download complete!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Base model download complete!


In [13]:
import os
from PIL import Image
from datasets import load_dataset

input_dir = "/kaggle/working/data/input"
os.makedirs(input_dir, exist_ok=True)

dataset_name = "Nechintosh/ghibli"
instance_prompt = "Ghibli anime style"

print(f"Loading HuggingFace dataset: {dataset_name}...")
try:
    hf_dataset = load_dataset(dataset_name, split="train")
except Exception as e:
    print(f"Failed to load HuggingFace dataset: {e}")
    hf_dataset = None
    if "numpy" in str(e) or "_blas_supports_fpe" in str(e) or "numpy._core._multiarray_umath" in str(e) or "dtype size" in str(e).lower() or "binary incompatibility" in str(e).lower():
        import sys
        import subprocess
        print("\nDETECTED NUMPY/BINARY COMPATIBILITY ISSUE. Fixing dynamically...")
        print("Reinstalling numpy, scipy, pandas, datasets, and pyarrow in sync...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "numpy==1.26.4", "scipy<1.13.0", "pandas<2.2.0", "pyarrow<15.0.0", "datasets"], check=True)
        print("Core packages successfully reinstalled and aligned with numpy 1.26.4!")
        print("CRITICAL: Please restart the notebook kernel / session (e.g. click 'Run' -> 'Restart Session' in Kaggle), then run the cells again.")

if hf_dataset is not None:
    print(f"Loaded dataset: {len(hf_dataset)} samples.")
    
    # Save the images and captions locally for diffusion-pipe
    for idx, item in enumerate(hf_dataset):
        image = item["image"]
        actual_description = item.get("text", item.get("caption", ""))
        
        # Clean whitespace and format the caption
        clean_text = " ".join(str(actual_description).split())
        if clean_text:
            caption = f"{instance_prompt}, {clean_text}"
        else:
            caption = f"{instance_prompt}"
        
        # Save image as .jpg
        image_path = os.path.join(input_dir, f"ghibli_{idx:03d}.jpg")
        image.convert("RGB").save(image_path)
        
        # Save corresponding caption as .txt
        txt_path = os.path.join(input_dir, f"ghibli_{idx:03d}.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(caption)
            
    print(f"Successfully prepared {len(hf_dataset)} image-caption pairs in {input_dir}!")
else:
    print("Cannot proceed: Dataset load failed.")


Loading HuggingFace dataset: Nechintosh/ghibli...


Resolving data files:   0%|          | 0/811 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/810 [00:00<?, ? examples/s]

Loaded dataset: 810 samples.
Successfully prepared 810 image-caption pairs in /kaggle/working/data/input!


In [15]:
os.makedirs("/kaggle/working/training", exist_ok=True)

dataset_toml_path = "/kaggle/working/training/dataset.toml"
config_toml_path = "/kaggle/working/training/config.toml"

dataset_content = """
resolutions = [512]
enable_ar_bucket = true
min_ar = 0.5
max_ar = 2.0
num_ar_buckets = 7
frame_buckets = [1, 33]

[[directory]]
path = '/kaggle/working/data/input'
num_repeats = 1
"""

config_content = """
output_dir = '/kaggle/working/data/output'
dataset = '/kaggle/working/training/dataset.toml'

# training settings
epochs = 5
micro_batch_size_per_gpu = 1
pipeline_stages = 1
gradient_accumulation_steps = 4
gradient_clipping = 1
warmup_steps = 50

# eval settings
eval_every_n_epochs = 5
eval_before_first_step = false
eval_micro_batch_size_per_gpu = 1
eval_gradient_accumulation_steps = 1

# misc settings
save_every_n_epochs = 1
checkpoint_every_n_minutes = 60
activation_checkpointing = true
partition_method = 'parameters'
save_dtype = 'bfloat16'
caching_batch_size = 1
steps_per_print = 5
blocks_to_swap = 28

[model]
type = 'wan'
ckpt_path = '/tmp/models/wan2.1'
dtype = 'bfloat16'
transformer_dtype = 'float8'
timestep_sample_method = 'logit_normal'

[adapter]
type = 'lora'
rank = 64
dtype = 'bfloat16'

[optimizer]
type = 'adamw8bit'
lr = 5e-5
betas = [0.9, 0.999]
weight_decay = 0.01
"""

with open(dataset_toml_path, "w", encoding="utf-8") as f:
    f.write(dataset_content.strip())

with open(config_toml_path, "w", encoding="utf-8") as f:
    f.write(config_content.strip())

print(f"Configurations saved:\n- {dataset_toml_path}\n- {config_toml_path}")


Configurations saved:
- /kaggle/working/training/dataset.toml
- /kaggle/working/training/config.toml


In [16]:
import os
import subprocess
import torch
import sys

# Environment variables for distributed memory-sharing stability
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["BNB_CUDA_VERSION"] = "121"

gpu_count = torch.cuda.device_count()
print(f"Found {gpu_count} GPU(s).")
print("\nNOTE: Checkpoints will be saved automatically every epoch.")
print("If you notice the loss stabilizing or you wish to stop training early, you can safely interrupt/stop this cell in Kaggle at any time!")
print("Any checkpoint directory saved prior to stopping remains fully intact.")
print("Running Cell 6 will detect and render test videos for all saved checkpoints so you can select the best one.\n")

# Programmatically pre-create cache metadata directories to avoid datasets.map FileNotFoundError in diffusion-pipe
try:
    import tomllib
    import numpy as np
    dataset_toml_path = "/kaggle/working/training/dataset.toml"
    if os.path.exists(dataset_toml_path):
        print("Pre-creating cache metadata directories...")
        with open(dataset_toml_path, "rb") as f:
            dataset_cfg = tomllib.load(f)
            
        enable_ar_bucket = dataset_cfg.get("enable_ar_bucket", False)
        min_ar = dataset_cfg.get("min_ar", 0.5)
        max_ar = dataset_cfg.get("max_ar", 2.0)
        num_ar_buckets = dataset_cfg.get("num_ar_buckets", 7)
        frame_buckets = dataset_cfg.get("frame_buckets", [1])
        if 1 not in frame_buckets:
            frame_buckets.append(1)
            
        if enable_ar_bucket:
            ars = np.geomspace(min_ar, max_ar, num=num_ar_buckets)
        else:
            ars = [1.0]
            
        ars = sorted(list(set(round(float(x), 3) for x in ars)))
        model_name = "wan"
        for directory in dataset_cfg.get("directory", []):
            dir_path = directory.get("path")
            if dir_path:
                cache_base = os.path.join(dir_path, "cache", model_name)
                for ar in ars:
                    for frames in frame_buckets:
                        dir_name = f"ar_frames_{ar:.3f}_{frames}"
                        metadata_dir = os.path.join(cache_base, dir_name, "metadata")
                        os.makedirs(metadata_dir, exist_ok=True)
                        print(f"  -> Created: {metadata_dir}")
except Exception as e:
    print(f"Warning: Failed to pre-create cache directories (will let train.py handle it): {e}")

# Launch deepspeed via python module runner (avoids PATH lookup issues on Kaggle/Colab)
deepspeed_base_cmd = [sys.executable, "-m", "deepspeed.launcher.runner"]

# Run deepspeed launcher pointing to diffusion-pipe/train.py
if gpu_count >= 2:
    print(f"Launching Parallel Multi-GPU ({gpu_count} GPUs) training using DeepSpeed...")
    cmd = deepspeed_base_cmd + [
        f"--num_gpus={gpu_count}",
        "diffusion-pipe/train.py",
        "--deepspeed",
        "--config", "/kaggle/working/training/config.toml"
    ]
else:
    print("Launching Single-GPU training using DeepSpeed...")
    cmd = deepspeed_base_cmd + [
        "diffusion-pipe/train.py",
        "--deepspeed",
        "--config", "/kaggle/working/training/config.toml"
    ]

# Stream outputs in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    print(f"\nTraining execution failed with exit code: {process.returncode}")
else:
    print("\nLoRA training completed successfully! Output saved in `/kaggle/working/data/output/`.")


Found 2 GPU(s).

NOTE: Checkpoints will be saved automatically every epoch.
If you notice the loss stabilizing or you wish to stop training early, you can safely interrupt/stop this cell in Kaggle at any time!
Any checkpoint directory saved prior to stopping remains fully intact.
Running Cell 6 will detect and render test videos for all saved checkpoints so you can select the best one.

Pre-creating cache metadata directories...
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_0.500_1/metadata
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_0.500_33/metadata
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_0.630_1/metadata
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_0.630_33/metadata
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_0.794_1/metadata
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_0.794_33/metadata
  -> Created: /kaggle/working/data/input/cache/wan/ar_frames_1.000_1/metadata
  -> Created: /kag

In [20]:
import os
import torch
import glob
import gc
import sys
import numpy as np

# Compatibility patch for accelerate when using numpy 1.26.x
if not hasattr(np.dtypes, "UInt32DType"):
    np.dtypes.UInt32DType = np.uint32

# Purge any half-initialized accelerate / diffusers modules from previous failed import attempts
for mod in list(sys.modules.keys()):
    if mod.startswith(("accelerate", "diffusers")):
        del sys.modules[mod]

from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from transformers import AutoTokenizer, UMT5EncoderModel

# Clear any existing variables from previous runs to free GPU memory
if 'pipe' in globals():
    print("Cleaning up previous pipeline from VRAM...")
    del pipe
if 'vae' in globals():
    del vae
if 'text_encoder' in globals():
    del text_encoder
gc.collect()
torch.cuda.empty_cache()

# Setup configuration
model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
video_prompt = (
    "Ghibli anime style, a cinematic, highly detailed video of a cozy cottage nestled in a green valley, "
    "fluffy white clouds, wind blowing through the grass, hand-painted aesthetic, masterpiece illustration, "
    "smooth natural motion, warm daylight, nostalgic vibes"
)
negative_prompt = (
    "worst quality, low quality, blurry, deformed, distorted, static, motionless, ugly, "
    "photorealistic, real life, 3d render, CGI, digital noise, watermark, text"
)

# 1. Load tokenizer and UMT5 text encoder on CPU to prevent VRAM OOM peak spikes
print("Loading tokenizer and UMT5 text encoder on CPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = UMT5EncoderModel.from_pretrained(
    model_id, 
    subfolder="text_encoder", 
    device_map="cpu",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
text_encoder.tie_weights()
# Manual fallback weight tying to guarantee embedding weights share memory
if hasattr(text_encoder, 'shared') and hasattr(text_encoder, 'encoder') and hasattr(text_encoder.encoder, 'embed_tokens'):
    text_encoder.encoder.embed_tokens.weight = text_encoder.shared.weight

# 2. Load VAE in float32 for stable decoding
print("Loading AutoencoderKLWan in float32 for video stabilization...")
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)

# 3. Load pipeline on GPU passing CPU-loaded text_encoder and tokenizer
print("Loading Wan2.1 T2V Diffusers Pipeline (embedding text_encoder on CPU)...")
pipe = WanPipeline.from_pretrained(
    model_id,
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    torch_dtype=torch.float16
)

# 4. Use the pipeline's official encode_prompt method on CPU to correctly format embeddings
print("Encoding prompts using pipeline on CPU...")
prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
    prompt=video_prompt,
    negative_prompt=negative_prompt,
    do_classifier_free_guidance=True,
    num_videos_per_prompt=1,
    device="cpu",
    dtype=torch.float16
)

# Move computed embeddings to GPU VRAM
prompt_embeds = prompt_embeds.to("cuda")
negative_prompt_embeds = negative_prompt_embeds.to("cuda")

# Print diagnostics
print(f"prompt_embeds: shape={prompt_embeds.shape}, mean={prompt_embeds.mean().item():.4f}, std={prompt_embeds.std().item():.4f}, has_nan={torch.isnan(prompt_embeds).any().item()}")
print(f"negative_prompt_embeds: shape={negative_prompt_embeds.shape}, mean={negative_prompt_embeds.mean().item():.4f}, std={negative_prompt_embeds.std().item():.4f}, has_nan={torch.isnan(negative_prompt_embeds).any().item()}")

# 5. Completely unload text encoder and tokenizer to keep CPU and GPU memory free
print("Unloading text encoder and tokenizer from pipeline...")
pipe.text_encoder = None
pipe.tokenizer = None
del text_encoder
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("Moving pipeline (Transformer and VAE) to GPU (VRAM)...")
pipe.to("cuda")
pipe.enable_attention_slicing()
if hasattr(pipe, "vae"):
    if hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()

# 3. Generate BEFORE Video
print(f"Generating BEFORE video (without LoRA)...")
generator = torch.Generator("cuda").manual_seed(42)
with torch.no_grad():
    video_frames_before = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        height=480,
        width=832,
        num_frames=21,
        num_inference_steps=50,
        guidance_scale=5.0,
        generator=generator
    ).frames[0]

before_path = "/kaggle/working/video_before_lora_ghibli.mp4"
after_path = "/kaggle/working/video_after_lora_ghibli.mp4"
export_to_video(video_frames_before, before_path, fps=8)
print(f"BEFORE video saved to: {before_path}")

# 4. Find and load the trained LoRA
output_glob = "/kaggle/working/data/output/*/epoch*"
matching_dirs = glob.glob(output_glob)

if not matching_dirs:
    print("ERROR: Could not find any epoch directories in `/kaggle/working/data/output/`!")
    print("Skipping AFTER video generation.")
else:
    lora_path = sorted(matching_dirs)[-1]
    print(f"Found LoRA adapter folder: {lora_path}")
    
    # Check if adapter model exists
    adapter_file = os.path.join(lora_path, "adapter_model.safetensors")
    if os.path.exists(adapter_file):
        print("Loading and fusing LoRA adapter weights...")
        pipe.load_lora_weights(lora_path, weight_name="adapter_model.safetensors")
        pipe.fuse_lora()
        
        # 5. Generate AFTER Video
        print(f"Generating AFTER video (with LoRA)...")
        generator = torch.Generator("cuda").manual_seed(42)
        with torch.no_grad():
            video_frames_after = pipe(
                prompt_embeds=prompt_embeds,
                negative_prompt_embeds=negative_prompt_embeds,
                height=480,
                width=832,
                num_frames=21,
                num_inference_steps=50,
                guidance_scale=5.0,
                generator=generator
            ).frames[0]
            
        export_to_video(video_frames_after, after_path, fps=8)
        print(f"AFTER video saved to: {after_path}")
    else:
        print(f"ERROR: adapter_model.safetensors not found in {lora_path}!")

# 6. Zip whatever videos were successfully generated for easy download
import zipfile
from IPython.display import FileLink

zip_path = "/kaggle/working/videos_comparison_ghibli.zip"
print(f"\nZipping generated videos to {zip_path}...")
zip_files_written = 0
with zipfile.ZipFile(zip_path, 'w') as zipf:
    if os.path.exists(before_path):
        zipf.write(before_path, os.path.basename(before_path))
        zip_files_written += 1
    if os.path.exists(after_path):
        zipf.write(after_path, os.path.basename(after_path))
        zip_files_written += 1

if zip_files_written > 0:
    print("\nDONE! Click the link below to download your generated videos:")
    display(FileLink("videos_comparison_ghibli.zip"))
else:
    print("\nNo videos were found to zip.")


This overrides automatic CUDA version selection.
If this was unintended clear the variable and retry: unset BNB_CUDA_VERSION

W0926 14:24:00.708000 58 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0926 14:24:00.776000 58 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Loading tokenizer and UMT5 text encoder on CPU...


tokenizer_config.json:   0%|          | 0.00/61.8k [00:00<?, ?B/s]

tokenizer/spiece.model: reconstructing file:   0%|          |  0.00B / 4.55MB            

tokenizer/spiece.model: downloading bytes:           |  0.00B            

...

  0%|          | 0/50 [00:00<?, ?it/s]

BEFORE video saved to: /kaggle/working/video_before_lora_ghibli.mp4
Found LoRA adapter folder: /kaggle/working/data/output/20260926_10-32-39/epoch5
Loading and fusing LoRA adapter weights...
Generating AFTER video (with LoRA)...


  0%|          | 0/50 [00:00<?, ?it/s]

AFTER video saved to: /kaggle/working/video_after_lora_ghibli.mp4

Zipping generated videos to /kaggle/working/videos_comparison_ghibli.zip...

DONE! Click the link below to download your generated videos:


/kaggle/working/videos_comparison_ghibli.zip